# Fase 1 — Pré-treinamento do Conector Multimodal

**Notebook:** `01_connector_pretraining.ipynb`  
**Fase:** 1 de 4  
**Projeto:** Arquitetura Híbrida Multimodal em torno de bitnet.cpp  

---

## Resumo

Este notebook implementa a **Fase 1** do pipeline: o pré-treinamento do conector MLP de duas camadas com ativação GeLU, conforme descrito na Seção 5.2 da especificação técnica. Durante esta fase, tanto o *encoder* da modalidade de entrada quanto o *backbone* textual permanecem **completamente congelados**. Apenas os parâmetros do conector são atualizados, utilizando pares de dados entrada–saída curtos (captioning ou instruction tuning).

Esta abordagem é coerente com o protocolo de treinamento do BitVLA, que demonstra que o alinhamento encoder–LLM pode ser estabelecido de forma eficiente quando o conector atua como único componente treinável em uma primeira fase.

---

## Índice

1. [Instalação de Dependências](#1-instalação-de-dependências)
2. [Configuração Global](#2-configuração-global)
3. [Montagem do Google Drive](#3-montagem-do-google-drive)
4. [Fundamentação Teórica](#4-fundamentação-teórica)
5. [Carregamento dos Componentes Congelados](#5-carregamento-dos-componentes-congelados)
6. [Instanciação do Conector](#6-instanciação-do-conector)
7. [Preparação dos Dados de Alinhamento](#7-preparação-dos-dados-de-alinhamento)
8. [Loop de Pré-treinamento do Conector](#8-loop-de-pré-treinamento-do-conector)
9. [Validação do Alinhamento](#9-validação-do-alinhamento)
10. [Persistência dos Artefatos](#10-persistência-dos-artefatos)
11. [Conclusões e Próximos Passos](#11-conclusões-e-próximos-passos)

## 1. Instalação de Dependências

In [ ]:
!pip install -q \
    transformers==4.44.0 \
    accelerate==0.33.0 \
    datasets==2.21.0 \
    sentencepiece==0.2.0 \
    safetensors==0.4.3 \
    einops==0.8.0 \
    timm==1.0.9

print("Instalação concluída.")

## 2. Configuração Global

In [ ]:
import json
import logging
import random
import sys
from pathlib import Path
from typing import Any, Dict, Optional

import numpy as np
import torch
import torch.nn as nn

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s — %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("phase1")

# Reprodutibilidade
SEED: int = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info("Dispositivo: %s", DEVICE)

# Constantes da fase
TEACHER_MODEL_ID: str     = "microsoft/bitnet-b1.58-2B-4T"
VISION_ENCODER_ID: str    = "openai/clip-vit-large-patch14"  # Substituir conforme encoder escolhido
DTYPE_HIGH: torch.dtype   = torch.bfloat16
D_ENC_VISION: int         = 1024   # Dimensão de saída do CLIP ViT-L
CONNECTOR_HIDDEN_DIM: int = 2048   # Dimensão intermediária do MLP conector
CONNECTOR_DROPOUT: float  = 0.0
LR_CONNECTOR: float       = 2e-4
N_EPOCHS: int             = 3
BATCH_SIZE: int           = 4
GRAD_CLIP: float          = 1.0
WARMUP_STEPS: int         = 100
LOG_INTERVAL: int         = 50
DRIVE_PROJECT_DIR: str    = "/content/drive/MyDrive/multimodal-ternary-llm"
PHASE_NAME: str           = "phase1_connector"

logger.info("Configuração da Fase 1 inicializada.")

## 3. Montagem do Google Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    logger.info("Google Drive montado.")
except ImportError:
    logger.warning("Ambiente não-Colab. Saltando montagem do Drive.")

CHECKPOINT_DIR = Path(DRIVE_PROJECT_DIR) / "checkpoints" / PHASE_NAME
LOG_DIR        = Path(DRIVE_PROJECT_DIR) / "logs"
METRICS_DIR    = Path(DRIVE_PROJECT_DIR) / "metrics"

# Carregar configurações da Fase 0
phase0_metrics_path = METRICS_DIR / "phase0_baseline_metrics.json"
PHASE0_METRICS: Dict[str, Any] = {}
if phase0_metrics_path.exists():
    with open(phase0_metrics_path) as f:
        PHASE0_METRICS = json.load(f)
    D_MODEL: int = PHASE0_METRICS["d_model"]
    logger.info("Métricas Fase 0 carregadas. d_model=%d", D_MODEL)
else:
    D_MODEL = 2048  # Fallback
    logger.warning("Métricas da Fase 0 não encontradas. Usando d_model=%d por padrão.", D_MODEL)

for d in (CHECKPOINT_DIR, LOG_DIR, METRICS_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 4. Fundamentação Teórica

### 4.1 Motivação para o Pré-treinamento Isolado do Conector

O conector MLP de duas camadas constitui o único elo entre o espaço de representação do encoder de modalidade e o espaço de embedding do backbone BitNet. O treinamento isolado deste componente — com encoder e backbone congelados — serve a dois propósitos:

1. **Eficiência computacional**: O número de parâmetros treináveis é minimizado, reduzindo o custo de cada *step* de gradiente e permitindo múltiplas iterações sobre os dados de alinhamento com recursos limitados (T4/L4).
2. **Estabilidade**: O backbone BitNet não é perturbado durante o aprendizado do mapeamento encoder→LLM, preservando as representações aprendidas na Fase 0.

### 4.2 Loss de Alinhamento na Fase 1

Nesta fase, utiliza-se exclusivamente a **language loss** (cross-entropy autoregressiva) sobre pares de dados multimodais curtos (e.g., imagem + legenda). A loss de distilação e a loss de alinhamento de representações são introduzidas apenas na Fase 2, quando blocos superiores do backbone são descongelados.

### 4.3 Manutenção de Precisão do Conector

O conector opera em BF16 ao longo de todas as fases, conforme especificado na Seção 3.3. Nenhuma quantização é aplicada a este componente, dado seu papel crítico como ponte de alinhamento e sua contribuição relativamente pequena ao tamanho total do modelo.

## 5. Carregamento dos Componentes Congelados

O encoder visual e o backbone textual são carregados e **completamente congelados** antes da instanciação do conector.

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    CLIPVisionModel,
    CLIPProcessor,
)

# ---------------------------------------------------------------------------
# Encoder visual (CLIP ViT-L como proxy; substituir por encoder da Qwen2.5-VL
# quando disponível no ambiente de produção)
# ---------------------------------------------------------------------------
logger.info("Carregando encoder visual: %s", VISION_ENCODER_ID)
vision_encoder = CLIPVisionModel.from_pretrained(
    VISION_ENCODER_ID, torch_dtype=DTYPE_HIGH
).to(DEVICE)
vision_processor = CLIPProcessor.from_pretrained(VISION_ENCODER_ID)

# Congelar encoder visual
for param in vision_encoder.parameters():
    param.requires_grad = False
vision_encoder.eval()
logger.info("Encoder visual congelado.")

# ---------------------------------------------------------------------------
# Backbone textual (BitNet / LLM teacher da Fase 0)
# ---------------------------------------------------------------------------
logger.info("Carregando backbone textual: %s", TEACHER_MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_ID, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

backbone = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_ID,
    torch_dtype=DTYPE_HIGH,
    device_map="auto",
    trust_remote_code=True,
)

# Congelar backbone completo
for param in backbone.parameters():
    param.requires_grad = False
backbone.eval()
logger.info("Backbone congelado.")

trainable_params = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
logger.info("Parâmetros treináveis no backbone: %d (esperado: 0)", trainable_params)
assert trainable_params == 0, "Backbone deve estar completamente congelado na Fase 1."

## 6. Instanciação do Conector

In [ ]:
import sys
sys.path.insert(0, "/content/multimodal-ternary-llm")  # Ajustar se necessário

# Alternativamente, definir o ModalityConnector inline para portabilidade no Colab
class ModalityConnector(nn.Module):
    """
    Two-layer GeLU MLP connector between a perception encoder and the BitNet core.

    Maintained in BF16 throughout all training phases. Not subjected to
    quantisation at any point, consistent with BitVLA training protocol.

    Parameters
    ----------
    d_enc : int
        Encoder output dimensionality.
    d_model : int
        BitNet backbone embedding dimensionality.
    d_hidden : int, optional
        MLP hidden dimension. Defaults to max(d_enc, d_model).
    dropout : float, optional
        Dropout probability after first projection. Default is 0.0.
    """

    def __init__(
        self,
        d_enc: int,
        d_model: int,
        d_hidden: Optional[int] = None,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        if d_hidden is None:
            d_hidden = max(d_enc, d_model)
        self.d_enc   = d_enc
        self.d_model = d_model

        self.proj_in  = nn.Linear(d_enc, d_hidden, bias=True)
        self.act      = nn.GELU()
        self.drop     = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.proj_out = nn.Linear(d_hidden, d_model, bias=True)

        nn.init.xavier_uniform_(self.proj_in.weight)
        nn.init.xavier_uniform_(self.proj_out.weight)
        nn.init.zeros_(self.proj_in.bias)
        nn.init.zeros_(self.proj_out.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Project encoder representations into the BitNet embedding space.

        Parameters
        ----------
        x : torch.Tensor
            Shape (batch, seq_len, d_enc).

        Returns
        -------
        torch.Tensor
            Shape (batch, seq_len, d_model).
        """
        x = x.to(next(self.parameters()).dtype)
        return self.proj_out(self.drop(self.act(self.proj_in(x))))


# Instanciar o conector
connector = ModalityConnector(
    d_enc=D_ENC_VISION,
    d_model=D_MODEL,
    d_hidden=CONNECTOR_HIDDEN_DIM,
    dropout=CONNECTOR_DROPOUT,
).to(DTYPE_HIGH).to(DEVICE)

trainable_connector = sum(p.numel() for p in connector.parameters() if p.requires_grad)
logger.info(
    "Conector instanciado. Parâmetros treináveis: %s",
    f"{trainable_connector:,}",
)

# Verificar que apenas o conector tem gradiente
assert trainable_connector > 0, "O conector deve ter parâmetros treináveis."

## 7. Preparação dos Dados de Alinhamento

Utiliza-se o dataset COCO Captions como proxy de dados imagem–texto para o pré-treinamento do conector. Em contextos de produção, substituir por dados de domínio específico.

In [ ]:
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import requests
from io import BytesIO

class ConnectorAlignmentDataset(Dataset):
    """
    Dataset for connector pre-training using image-caption pairs.

    Encodes images via the frozen vision encoder and tokenises captions
    for language modelling supervision.

    Parameters
    ----------
    hf_dataset : datasets.Dataset
        HuggingFace dataset with 'image' and 'caption' fields.
    vision_processor : CLIPProcessor
        Processor for the vision encoder.
    tokenizer : PreTrainedTokenizer
        Tokenizer for the language backbone.
    max_text_len : int, optional
        Maximum caption token length. Default is 128.
    """

    def __init__(self, hf_dataset, vision_processor, tokenizer, max_text_len: int = 128):
        self.dataset          = hf_dataset
        self.vision_processor = vision_processor
        self.tokenizer        = tokenizer
        self.max_text_len     = max_text_len

    def __len__(self) -> int:
        return len(self.dataset)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        item   = self.dataset[idx]
        image  = item["image"].convert("RGB")
        # Primeiro elemento da lista de captions
        caption = item["captions"][0] if isinstance(item.get("captions"), list) else item.get("caption", "")

        pix = self.vision_processor(images=image, return_tensors="pt")["pixel_values"].squeeze(0)
        tok = self.tokenizer(
            caption,
            truncation=True,
            max_length=self.max_text_len,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "pixel_values": pix,
            "input_ids":    tok["input_ids"].squeeze(0),
            "attention_mask": tok["attention_mask"].squeeze(0),
        }


logger.info("Carregando dataset de alinhamento (COCO Captions — split train[:5000])...")
raw_dataset = load_dataset("nlphuji/flickr30k", split="test[:2000]")  # Proxy leve para Colab

train_ds = ConnectorAlignmentDataset(
    hf_dataset=raw_dataset,
    vision_processor=vision_processor,
    tokenizer=tokenizer,
    max_text_len=128,
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
logger.info("Dataset pronto. N amostras: %d | N batches/epoch: %d", len(train_ds), len(train_loader))

## 8. Loop de Pré-treinamento do Conector

In [ ]:
import math
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
import torch.nn.functional as F

# ---------------------------------------------------------------------------
# Optimizador (somente parâmetros do conector)
# ---------------------------------------------------------------------------
optimiser = AdamW(connector.parameters(), lr=LR_CONNECTOR, weight_decay=1e-2, betas=(0.9, 0.95))

total_steps      = N_EPOCHS * len(train_loader)
warmup_scheduler = LinearLR(optimiser, start_factor=1e-3, end_factor=1.0, total_iters=WARMUP_STEPS)
cosine_scheduler = CosineAnnealingLR(optimiser, T_max=max(total_steps - WARMUP_STEPS, 1), eta_min=1e-6)
scheduler        = SequentialLR(optimiser, [warmup_scheduler, cosine_scheduler], milestones=[WARMUP_STEPS])

logger.info(
    "Optimizador: AdamW | lr=%.2e | total_steps=%d | warmup=%d",
    LR_CONNECTOR, total_steps, WARMUP_STEPS,
)

# ---------------------------------------------------------------------------
# Loop de treinamento
# ---------------------------------------------------------------------------
phase1_metrics: list[Dict[str, float]] = []
global_step: int = 0

for epoch in range(N_EPOCHS):
    connector.train()
    epoch_loss: float = 0.0
    n_batches: int    = 0

    for step, batch in enumerate(train_loader):
        pixel_values  = batch["pixel_values"].to(DEVICE, dtype=DTYPE_HIGH)
        input_ids     = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        # --- Encoder visual (congelado, sem gradiente) ---
        with torch.no_grad():
            visual_feats = vision_encoder(
                pixel_values=pixel_values
            ).last_hidden_state  # (B, n_patches+1, d_enc)

        # --- Projeção via conector (treinável) ---
        projected = connector(visual_feats)  # (B, n_patches+1, d_model)

        # --- Obter embeddings de texto do backbone (congelado) ---
        with torch.no_grad():
            text_embeds = backbone.get_input_embeddings()(input_ids)  # (B, L, d_model)

        # Concatenar visual projetado + texto: [visual | texto]
        combined_embeds = torch.cat([projected, text_embeds], dim=1)  # (B, n_vis+L, d_model)
        combined_mask   = torch.cat([
            torch.ones(projected.shape[:2], device=DEVICE, dtype=attention_mask.dtype),
            attention_mask,
        ], dim=1)

        # Labels: ignorar tokens visuais (-100), supervisionar apenas tokens de texto
        n_vis  = projected.shape[1]
        labels = torch.cat([
            torch.full((input_ids.shape[0], n_vis), -100, device=DEVICE),
            input_ids,
        ], dim=1)

        # --- Forward pelo backbone com embeddings de entrada ---
        with torch.no_grad():
            outputs = backbone(
                inputs_embeds=combined_embeds,
                attention_mask=combined_mask,
            )

        # Language loss: cross-entropy autoregressiva sobre tokens de texto
        logits      = outputs.logits
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        loss = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            ignore_index=-100,
        )

        optimiser.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(connector.parameters(), GRAD_CLIP)
        optimiser.step()
        scheduler.step()

        epoch_loss  += loss.item()
        n_batches   += 1
        global_step += 1

        if (step + 1) % LOG_INTERVAL == 0:
            avg_loss = epoch_loss / n_batches
            lr       = scheduler.get_last_lr()[0]
            logger.info(
                "Epoch %d | Step %d | loss=%.4f | lr=%.2e",
                epoch + 1, step + 1, avg_loss, lr,
            )

    epoch_avg = epoch_loss / max(n_batches, 1)
    phase1_metrics.append({"epoch": epoch + 1, "avg_loss": epoch_avg})
    logger.info("Epoch %d concluída. Avg loss: %.4f", epoch + 1, epoch_avg)

logger.info("Pré-treinamento do conector concluído.")

## 9. Validação do Alinhamento

In [ ]:
# ---------------------------------------------------------------------------
# Verificação de normalidade da distribuição de saída do conector
# ---------------------------------------------------------------------------
connector.eval()

sample_batch = next(iter(DataLoader(train_ds, batch_size=8)))
with torch.no_grad():
    pv = sample_batch["pixel_values"].to(DEVICE, dtype=DTYPE_HIGH)
    vis_feats = vision_encoder(pixel_values=pv).last_hidden_state
    projected_sample = connector(vis_feats)

output_mean = projected_sample.mean().item()
output_std  = projected_sample.std().item()
output_max  = projected_sample.abs().max().item()

logger.info(
    "Distribuição da saída do conector — mean=%.4f | std=%.4f | abs_max=%.4f",
    output_mean, output_std, output_max,
)

# Critério informal: desvio padrão deve ser não-trivial para detectar colapso
assert output_std > 1e-4, (
    f"Possível colapso detectado: std da saída do conector = {output_std:.6f} ≈ 0."
)
logger.info("Validação de alinhamento: PASSADA.")

print(f"\nSaída do conector — média: {output_mean:.4f} | desvio: {output_std:.4f}")

## 10. Persistência dos Artefatos

In [ ]:
# Salvar pesos do conector
connector_ckpt_path = CHECKPOINT_DIR / "connector_phase1.pt"
torch.save({
    "model_state_dict": connector.state_dict(),
    "optimiser_state_dict": optimiser.state_dict(),
    "epoch": N_EPOCHS,
    "d_enc": D_ENC_VISION,
    "d_model": D_MODEL,
    "d_hidden": CONNECTOR_HIDDEN_DIM,
}, connector_ckpt_path)
logger.info("Conector salvo em: %s", connector_ckpt_path)

# Salvar métricas
metrics_path = METRICS_DIR / "phase1_metrics.json"
with open(metrics_path, "w") as f:
    json.dump({"training_loss_per_epoch": phase1_metrics}, f, indent=2)
logger.info("Métricas salvas em: %s", metrics_path)

print("\nArtefatos da Fase 1 persistidos com sucesso.")

## 11. Conclusões e Próximos Passos

### Resultados da Fase 1

O conector MLP de duas camadas foi pré-treinado com encoder visual e backbone textual completamente congelados. Os artefatos produzidos são:

| Artefato | Localização |
|---|---|
| Pesos do conector | `checkpoints/phase1_connector/connector_phase1.pt` |
| Métricas de treinamento | `metrics/phase1_metrics.json` |

### Próxima Fase

Prosseguir para `02_multimodal_alignment.ipynb` (**Fase 2**), que introduz descongelamento seletivo dos blocos superiores do backbone BitNet, distillation loss do teacher e representation alignment loss, conforme a Seção 5.3 da especificação técnica.